# Friendly Speed Insights with Gemini (Colab版)

Google PageSpeed Insightsの診断結果とHTMLソースコードをAI（Gemini）が分析し、具体的な改善案を提案します。
ブラウザ版で発生するCORSエラーを回避するため、Python環境で実行します。

In [ ]:
# 必要なライブラリのインストール
!pip install -q -U google-generativeai requests beautifulsoup4

In [ ]:
import requests
import google.generativeai as genai
from bs4 import BeautifulSoup
from IPython.display import display, Markdown
import ipywidgets as widgets

# --- 設定 ---

# Gemini API Keyを入力
GEMINI_API_KEY = "" #@param {type:"string"}

# 診断したいURL
TARGET_URL = "https://example.com" #@param {type:"string"}

# --- ロジック ---

def run_analysis(api_key, url):
    if not api_key or not url:
        print("API KeyとURLを入力してください。")
        return

    genai.configure(api_key=api_key)

    print(f"🔍 PageSpeed Insightsで {url} を分析中...")

    # 1. PageSpeed Insights API (Mobile)
    psi_url = f"https://www.googleapis.com/pagespeedonline/v5/runPagespeed?url={url}&strategy=mobile"
    try:
        resp = requests.get(psi_url)
        data = resp.json()

        if "error" in data:
            print(f"⚠️ PSI Error: {data['error']['message']}")
            return

        # スコア取得
        score = int(data["lighthouseResult"]["categories"]["performance"]["score"] * 100)
        audits = data["lighthouseResult"]["audits"]

        # 改善インパクトのある項目を抽出
        target_audits = [
            'modern-image-formats',
            'uses-optimized-images',
            'offscreen-images',
            'unused-javascript',
            'server-response-time',
            'render-blocking-resources'
        ]
        
        failed_audits = []
        for audit_id in target_audits:
            if audit_id in audits:
                audit = audits[audit_id]
                savings = audit.get('details', {}).get('overallSavingsMs', 0)
                if savings > 0 or audit.get('score', 1) < 1:
                    failed_audits.append({
                        "title": audit["title"],
                        "savings": f"{savings/1000:.2f}秒" if savings > 0 else "改善推奨"
                    })

    except Exception as e:
        print(f"❌ PSI API Error: {e}")
        return

    print(f"📄 HTMLソースを取得中...")

    # 2. HTML取得
    html_content = "(HTML取得失敗)"
    try:
        # User-Agentを偽装しないと拒否されることがあるため一般的なものを指定
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        res = requests.get(url, headers=headers, timeout=10)
        if res.status_code == 200:
            # トークン節約のためbody中心に抜粋するか、単純に先頭を切り出す
            text = res.text
            html_content = text[:30000] # 先頭30000文字
    except Exception as e:
        print(f"⚠️ HTML Fetch Warning: {e}")

    print(f"🤖 Gemini 2.5 Flashで解析中... (Score: {score})")

    # 3. Gemini分析
    prompt = f"""
あなたは世界最高峰のWebパフォーマンス改善エンジニアです。
以下のPageSpeed Insights診断結果と、WebページのHTMLソースコードを分析し、
具体的で実行可能な改善アドバイスを日本語で提供してください。

【診断対象】
URL: {url}
パフォーマンススコア: {score}/100

【優先的に改善すべき項目】
{chr(10).join([f'- {a['title']} ({a['savings']})' for a in failed_audits])}

【HTMLソースコード（抜粋）】
```html
{html_content}
```

【指示】
1. **HTMLコードに基づく具体的な指摘**: 提供されたHTMLを見て、「この<img>タグにloading="lazy"がない」「head内のこのスクリプトがレンダリングをブロックしている」など、コードレベルで指摘してください。
2. **修正前後のコード例**: 「修正前」と「修正後」のコード例を示して説明してください。
3. **初心者への配慮**: 専門用語には簡単な解説を添えてください。
"""

    try:
        model = genai.GenerativeModel('gemini-2.0-flash-exp') # または gemini-1.5-flash
        response = model.generate_content(prompt)
        
        display(Markdown(f"## 📊 分析結果 (Score: {score})"))
        display(Markdown(response.text))

    except Exception as e:
        print(f"❌ Gemini Error: {e}")
        print("Gemini 2.0 Flash Expが使えない場合は、コード内のモデル名を 'gemini-1.5-flash' に変更してください。")

# 実行ボタン
button = widgets.Button(description="診断開始")
output = widgets.Output()

def on_button_clicked(b):
    with output:
        output.clear_output()
        run_analysis(GEMINI_API_KEY, TARGET_URL)

button.on_click(on_button_clicked)
display(button, output)